In [2]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, torch

print(subprocess.getoutput('nvidia-smi'))
print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
    print('✅ cudnn.benchmark = True')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Mar 11 17:37:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|          

In [3]:
import os

DRIVE_ROOT     = '/content/drive/MyDrive/ChagaSight'
DATA_DIR       = f'{DRIVE_ROOT}/data/processed/1d_signals.100hz'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CONFIG = {
    'data_dir'                 : DATA_DIR,
    'checkpoint_dir'           : CHECKPOINT_DIR,
    'subset'                   : 1.0,
    'epochs'                   : 30,
    'batch_size'               : 64,        # T4=64 | A100=128
    'num_workers'              : 2,
    'lr'                       : 1.5e-4,
    'warmup_epochs'            : 2,
    'mask_ratio'               : 0.75,
    'save_every'               : 5,
    'checkpoint_every_batches' : 300,
    'use_amp'                  : True,
}

print('✅ Configuration:')
for k, v in CONFIG.items():
    print(f'   {k:<30} : {v}')
print(f'\n📂 Data        : {DATA_DIR}')
print(f'💾 Checkpoints : {CHECKPOINT_DIR}')

✅ Configuration:
   data_dir                       : /content/drive/MyDrive/ChagaSight/data/processed/1d_signals.100hz
   checkpoint_dir                 : /content/drive/MyDrive/ChagaSight/checkpoints
   subset                         : 1.0
   epochs                         : 30
   batch_size                     : 64
   num_workers                    : 2
   lr                             : 0.00015
   warmup_epochs                  : 2
   mask_ratio                     : 0.75
   save_every                     : 5
   checkpoint_every_batches       : 300
   use_amp                        : True

📂 Data        : /content/drive/MyDrive/ChagaSight/data/processed/1d_signals.100hz
💾 Checkpoints : /content/drive/MyDrive/ChagaSight/checkpoints


In [4]:
import subprocess, torch
import numpy as np
from pathlib import Path

print('=' * 65)
print('🔍  PRE-FLIGHT CHECK')
print('=' * 65)

DATA_PATH = Path(CONFIG['data_dir'])
CKPT_PATH = Path(CONFIG['checkpoint_dir'])
DATASETS  = ['ptbxl', 'samitrop', 'code15']


def count_npy(folder: Path) -> tuple:
    """
    Count .npy files using multiple strategies, no timeout.
    Returns (count, method) where count=-1 means missing, -2 means failed.
    """
    if not folder.exists():
        return -1, 'missing'

    # Strategy 1: ls glob (fastest for flat large folders)
    try:
        r = subprocess.run(
            f"ls -1 '{folder}'/*.npy 2>/dev/null | wc -l",
            shell=True, capture_output=True, text=True
        )
        n = int(r.stdout.strip() or 0)
        if n > 0:
            return n, 'ls_glob'
    except Exception:
        pass

    # Strategy 2: find (no timeout — let it run as long as needed)
    try:
        r = subprocess.run(
            f"find '{folder}' -maxdepth 1 -name '*.npy' -type f | wc -l",
            shell=True, capture_output=True, text=True
        )
        n = int(r.stdout.strip() or 0)
        if n >= 0:
            return n, 'find'
    except Exception:
        pass

    # Strategy 3: ls pipe grep
    try:
        r = subprocess.run(
            f"ls -1 '{folder}' | grep -c '\.npy$'",
            shell=True, capture_output=True, text=True
        )
        n = int(r.stdout.strip() or 0)
        return n, 'ls_grep'
    except Exception:
        pass

    return -2, 'all_failed'


def get_sample_files(folder: Path, n: int = 3) -> list:
    """Get a few sample filenames — proves files exist even if count fails."""
    try:
        r = subprocess.run(
            f"ls -1 '{folder}'/*.npy 2>/dev/null | head -{n}",
            shell=True, capture_output=True, text=True
        )
        return [l.strip() for l in r.stdout.strip().split('\n') if l.strip()]
    except Exception:
        return []


# ── 1. Data directory ──────────────────────────────────────────
print(f'\n📁 1. Data directory')
if not DATA_PATH.exists():
    print(f'   ❌ MISSING: {DATA_PATH}')
else:
    print(f'   ✅ {DATA_PATH}')

# ── 2. Per-dataset counts ──────────────────────────────────────
print(f'\n📂 2. Dataset file counts:')
all_ok    = True
ds_counts = {}

for ds in DATASETS:
    ds_path = DATA_PATH / ds
    print(f'\n   🔍 {ds}', flush=True)

    if not ds_path.exists():
        print(f'      ❌ Folder not found: {ds_path}')
        all_ok = False
        ds_counts[ds] = 0
        continue

    print(f'      Counting...', end='', flush=True)
    count, method = count_npy(ds_path)
    print(f'\r', end='')

    if count == -1:
        print(f'      ❌ Folder missing')
        all_ok = False
        ds_counts[ds] = 0

    elif count == -2 or count == 0:
        # Count failed or returned 0 — check if files actually exist
        samples = get_sample_files(ds_path, n=3)
        if samples:
            print(f'      ⚠️  Count returned 0 but files EXIST (Drive indexing lag)')
            print(f'         Sample: {Path(samples[0]).name}')
            print(f'         Training will work — dataset loads files directly')
            ds_counts[ds] = -3   # special: exists but uncountable right now
        else:
            print(f'      ❌ 0 files and no samples accessible')
            print(f'         → Check preprocessing or remount Drive')
            all_ok = False
            ds_counts[ds] = 0

    else:
        print(f'      ✅ {count:,} files  (via {method})')
        # Quick load test
        samples = get_sample_files(ds_path, n=1)
        if samples:
            try:
                arr = np.load(samples[0])
                shape_ok = arr.shape == (12, 1000)
                print(f'      ✅ Sample OK: shape={arr.shape} dtype={arr.dtype} '
                      f'{"✅" if shape_ok else "⚠️ expected (12,1000)"}')
            except Exception as e:
                print(f'      ⚠️  Sample load error: {e}')
        ds_counts[ds] = count

# ── 3. code15 extra check ─────────────────────────────────────
print(f'\n🔍 3. code15 detailed check:')
c15 = DATA_PATH / 'code15'
if c15.exists():
    # Check it's flat (no subfolders — you confirmed this)
    r = subprocess.run(
        f"find '{c15}' -maxdepth 1 -type d | wc -l",
        shell=True, capture_output=True, text=True
    )
    n_dirs = max(0, int(r.stdout.strip() or 1) - 1)
    print(f'   Subfolders inside code15: {n_dirs}  '
          f'{"✅ flat (expected)" if n_dirs == 0 else "⚠️ has subfolders"}')

    samples = get_sample_files(c15, n=5)
    if samples:
        print(f'   Files accessible: ✅')
        for s in samples[:3]:
            print(f'     → {Path(s).name}')
        try:
            arr = np.load(samples[0])
            print(f'   Direct load: ✅ shape={arr.shape} range=[{arr.min():.2f},{arr.max():.2f}]')
            print(f'   NaN={np.isnan(arr).any()} Inf={np.isinf(arr).any()}')
        except Exception as e:
            print(f'   Load error: {e}')
    else:
        print(f'   ❌ Cannot access files')
        all_ok = False

# ── 4. Checkpoint ─────────────────────────────────────────────
print(f'\n💾 4. Checkpoint status:')
ckpt_file = CKPT_PATH / 'stmem_1d_checkpoint.pt'
best_file  = CKPT_PATH / 'stmem_1d_pretrained.pt'

if ckpt_file.exists():
    ckpt      = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    epoch     = ckpt['epoch']
    batch     = ckpt.get('batch_idx', 0)
    complete  = ckpt.get('epoch_complete', True)
    best_loss = ckpt['best_loss']
    ts        = ckpt.get('timestamp', '?')

    print(f'   ✅ Rolling checkpoint found')
    print(f'   Epoch: {epoch+1}  |  Best loss: {best_loss:.4f}  |  Saved: {ts}')

    pos_shape = ckpt['model_state_dict']['pos_embed'].shape
    compat = '✅ (241 = CLS + 240 patches)' if pos_shape[1] == 241 \
             else '⚠️  Unexpected — check architecture!'
    print(f'   pos_embed shape: {pos_shape}  {compat}')

    if complete:
        print(f'\n   ▶️  Will START epoch {epoch+2}')
    else:
        print(f'\n   ▶️  Will RESUME epoch {epoch+1} from batch {batch+1}')
else:
    print('   ℹ️  No checkpoint — will start fresh from epoch 1')

if best_file.exists():
    best = torch.load(best_file, map_location='cpu', weights_only=False)
    sz   = best_file.stat().st_size / 1e6
    print(f'\n   🏆 Best model: loss={best["loss"]:.4f}  epoch={best["epoch"]+1}  size={sz:.1f}MB')

# ── 5. Verdict ────────────────────────────────────────────────
code15_ok = ds_counts.get('code15', 0) != 0
print('\n' + '=' * 65)
if all_ok or (ds_counts.get('ptbxl', 0) > 0
              and ds_counts.get('samitrop', 0) > 0
              and code15_ok):
    print('✅ ALL CHECKS PASSED — safe to run training!')
else:
    print('❌ ISSUES FOUND — fix before training!')
print('=' * 65)

🔍  PRE-FLIGHT CHECK

📁 1. Data directory


<>:49: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_888/1074259633.py:49: SyntaxWarning: invalid escape sequence '\.'
  f"ls -1 '{folder}' | grep -c '\.npy$'",


   ✅ /content/drive/MyDrive/ChagaSight/data/processed/1d_signals.100hz

📂 2. Dataset file counts:

   🔍 ptbxl
      ✅ 21,799 files  (via ls_glob)
      ✅ Sample OK: shape=(12, 1000) dtype=float32 ✅

   🔍 samitrop
      ✅ 1,633 files  (via ls_glob)
      ✅ Sample OK: shape=(12, 1000) dtype=float32 ✅

   🔍 code15
      ❌ 0 files and no samples accessible
         → Check preprocessing or remount Drive

🔍 3. code15 detailed check:
   Subfolders inside code15: 0  ✅ flat (expected)
   ❌ Cannot access files

💾 4. Checkpoint status:
   ℹ️  No checkpoint — will start fresh from epoch 1

❌ ISSUES FOUND — fix before training!


In [5]:
import subprocess
import numpy as np
from pathlib import Path

N_SAMPLE     = 30
EXPECT_SHAPE = (12, 1000)

DATA_PATH = Path(CONFIG['data_dir'])
DATASETS  = ['ptbxl', 'samitrop', 'code15']

print('=' * 65)
print('🔬  DEEP DATA VALIDATOR')
print(f'   Checking {N_SAMPLE} random files per dataset')
print(f'   Expected shape: {EXPECT_SHAPE}')
print('=' * 65)

all_passed = True


def get_random_sample(folder: Path, n: int = 30) -> tuple:
    """
    Returns (sample_paths, total_count) using shell — no timeout.
    Shell streams 300k lines; Python only receives N back.
    """
    # Random sample via shuf
    try:
        r = subprocess.run(
            f"find '{folder}' -maxdepth 1 -name '*.npy' -type f | shuf -n {n}",
            shell=True, capture_output=True, text=True
        )
        sample = [l for l in r.stdout.strip().split('\n') if l.strip()]
    except Exception:
        # fallback: ls head
        r = subprocess.run(
            f"ls -1 '{folder}'/*.npy 2>/dev/null | head -{n}",
            shell=True, capture_output=True, text=True
        )
        sample = [l.strip() for l in r.stdout.strip().split('\n') if l.strip()]

    # Total count
    try:
        r2 = subprocess.run(
            f"ls -1 '{folder}'/*.npy 2>/dev/null | wc -l",
            shell=True, capture_output=True, text=True
        )
        total = int(r2.stdout.strip() or len(sample))
    except Exception:
        total = len(sample)

    return sample, total


for ds in DATASETS:
    ds_path = DATA_PATH / ds
    print(f'\n📂 {ds}')

    if not ds_path.exists():
        print(f'   ❌ Folder not found')
        all_passed = False
        continue

    sample, total = get_random_sample(ds_path, n=N_SAMPLE)

    if not sample:
        print(f'   ❌ No files accessible — check Drive mount')
        all_passed = False
        continue

    print(f'   Total  : ~{total:,} files')
    print(f'   Sampled: {len(sample)} files for validation')

    shapes, dtypes = [], []
    nan_count = inf_count = err_count = 0

    for path in sample:
        try:
            arr = np.load(path)
            shapes.append(arr.shape)
            dtypes.append(str(arr.dtype))
            if np.isnan(arr).any():  nan_count += 1
            if np.isinf(arr).any():  inf_count += 1
        except Exception as ex:
            print(f'   ❌ Load error: {Path(path).name} → {ex}')
            err_count  += 1
            all_passed  = False

    n_ok          = len(sample) - err_count
    unique_shapes = set(shapes)
    unique_dtypes = set(dtypes)

    shape_ok = unique_shapes == {EXPECT_SHAPE}
    dtype_ok = all(d in ('float32', 'float64') for d in unique_dtypes)
    nan_ok   = nan_count == 0
    inf_ok   = inf_count == 0

    print(f'   Shapes : {unique_shapes}  {"✅" if shape_ok else "⚠️  expected (12,1000)"}')
    print(f'   Dtypes : {unique_dtypes}  {"✅" if dtype_ok else "⚠️  expected float32"}')
    print(f'   NaN    : {nan_count}/{n_ok}  {"✅" if nan_ok else "⚠️  NaNs found!"}')
    print(f'   Inf    : {inf_count}/{n_ok}  {"✅" if inf_ok else "⚠️  Infs found!"}')
    print(f'   Errors : {err_count}/{len(sample)}  {"✅" if err_count==0 else "❌"}')

    if shape_ok and dtype_ok and nan_ok and inf_ok and err_count == 0:
        print(f'   ✅ {ds} — all checks passed!')
    else:
        all_passed = False

print('\n' + '=' * 65)
print('✅ All datasets validated!' if all_passed else '❌ Issues found — fix before training!')
print('=' * 65)

🔬  DEEP DATA VALIDATOR
   Checking 30 random files per dataset
   Expected shape: (12, 1000)

📂 ptbxl
   Total  : ~21,799 files
   Sampled: 30 files for validation


KeyboardInterrupt: 

In [ ]:
import os, subprocess
import numpy as np
import torch
import torch.nn as nn
import signal
from pathlib import Path
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time


# ── Drive-safe file lister ────────────────────────────────────────
def list_npy_drive(folder: Path) -> list:
    """
    List ALL .npy files via subprocess find — no timeout.
    Shell streams the output; Python reads it as one string after.
    For 342k files over FUSE this takes ~2-3 min — done once at init.
    """
    try:
        result = subprocess.run(
            ['find', str(folder), '-maxdepth', '1', '-name', '*.npy', '-type', 'f'],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            return []
        return [l for l in result.stdout.strip().split('\n') if l.strip()]
    except OSError:
        return []


# ── Dataset ───────────────────────────────────────────────────────
class STMEMSignalDataset(Dataset):
    """
    Loads all .npy ECG signals from ptbxl/, samitrop/, code15/.
    File listing uses subprocess (Drive-safe for 300k+ files).
    Path collection for code15 takes ~2-3 min over FUSE — once only.
    """
    DATASETS = ['ptbxl', 'samitrop', 'code15']

    def __init__(self, data_dir, subset=1.0, seed=42):
        self.data_dir     = Path(data_dir)
        self.signal_paths = []

        for ds in self.DATASETS:
            ds_path = self.data_dir / ds

            if not ds_path.exists():
                print(f'   ⚠️  {ds}: NOT FOUND — skipped')
                continue

            note = ' (collecting ~342k paths, ~2-3 min)' if ds == 'code15' else ''
            print(f'   📁 {ds:<12}{note}', end='', flush=True)

            paths = list_npy_drive(ds_path)
            n     = len(paths)

            if n == 0:
                print(f'\n   ⚠️  {ds}: 0 files returned — '
                      f'try drive.mount("/content/drive", force_remount=True)')
                continue

            self.signal_paths.extend([Path(p) for p in paths])
            print(f' → {n:,} files')

        if not self.signal_paths:
            raise ValueError(
                f'No .npy files found under {data_dir}. '
                f'Expected subfolders: {self.DATASETS}'
            )

        if subset < 1.0:
            rng = np.random.RandomState(seed)
            n   = int(len(self.signal_paths) * subset)
            idx = rng.choice(len(self.signal_paths), n, replace=False)
            self.signal_paths = [self.signal_paths[i] for i in idx]

        print(f'\n   ✅ Total: {len(self.signal_paths):,} signals ({subset*100:.0f}%)')

    def __len__(self):
        return len(self.signal_paths)

    def __getitem__(self, idx):
        return torch.from_numpy(np.load(self.signal_paths[idx])).float()


# ── Patch Embedding ───────────────────────────────────────────────
class PatchEmbed1D(nn.Module):
    """
    Input : (B, 12, 1000)
    Output: (B, 240, embed_dim)   [12 leads × 20 patches]
    """
    def __init__(self, num_leads=12, seq_len=1000, patch_size=50, embed_dim=768):
        super().__init__()
        self.num_leads            = num_leads
        self.patch_size           = patch_size
        self.num_patches_per_lead = seq_len // patch_size   # 20
        self.num_patches          = num_leads * self.num_patches_per_lead  # 240

        self.proj       = nn.Conv1d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.lead_embed = nn.Parameter(torch.zeros(1, num_leads, 1, embed_dim))
        nn.init.trunc_normal_(self.lead_embed, std=0.02)

    def forward(self, x):
        B, L, T = x.shape
        x = x.view(B * L, 1, T)
        x = self.proj(x)
        x = x.view(B, L, -1, self.num_patches_per_lead).permute(0, 1, 3, 2)
        x = x + self.lead_embed
        return x.reshape(B, self.num_patches, -1)


# ── ST-MEM Model ──────────────────────────────────────────────────
class STMEM1D(nn.Module):
    """
    ST-MEM 1D — matches local vit_1d_fm.py exactly.

    Architecture:
      pos_embed  : (1, 241, D)  — CLS + 240 patches
      sep_tokens : defined, NOT used in forward (matches original)
      Encoder    : 12× TransformerEncoderLayer (norm_first=True)
      Decoder    : 4×  TransformerEncoderLayer (embed_dim=512)
      Loss       : MSE on masked patches only (75% masked)
    """
    def __init__(self, embed_dim=768, depth=12, num_heads=12,
                 decoder_embed_dim=512, decoder_depth=4,
                 decoder_num_heads=8, mask_ratio=0.75):
        super().__init__()

        self.patch_embed = PatchEmbed1D(embed_dim=embed_dim)
        num_patches      = self.patch_embed.num_patches   # 240
        self.num_leads   = self.patch_embed.num_leads
        self.mask_ratio  = mask_ratio

        # Encoder
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed  = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))  # 241
        self.sep_tokens = nn.Parameter(torch.zeros(1, self.num_leads - 1, embed_dim))  # defined; not used in forward

        self.encoder = nn.ModuleList([
            nn.TransformerEncoderLayer(
                embed_dim, num_heads, embed_dim * 4,
                dropout=0.0, activation='gelu',
                batch_first=True, norm_first=True)
            for _ in range(depth)
        ])
        self.encoder_norm = nn.LayerNorm(embed_dim)

        # Decoder
        self.decoder_embed     = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token        = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim))  # 241

        self.decoder = nn.ModuleList([
            nn.TransformerEncoderLayer(
                decoder_embed_dim, decoder_num_heads, decoder_embed_dim * 4,
                dropout=0.0, activation='gelu',
                batch_first=True, norm_first=True)
            for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, self.patch_embed.patch_size)

        for p in [self.cls_token, self.pos_embed, self.sep_tokens,
                  self.mask_token, self.decoder_pos_embed]:
            nn.init.trunc_normal_(p, std=0.02)

    def forward(self, signals):
        B = signals.shape[0]
        x = self.patch_embed(signals)               # (B, 240, D)

        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1) + self.pos_embed  # (B, 241, D)

        N           = x.shape[1] - 1                # 240
        num_masked  = int(N * self.mask_ratio)       # 180
        noise       = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep    = ids_shuffle[:, :N - num_masked]  # 60 visible

        x_vis = torch.gather(
            x[:, 1:], 1,
            ids_keep.unsqueeze(-1).expand(-1, -1, x.shape[-1])
        )
        x_vis = torch.cat([x[:, :1], x_vis], dim=1)   # (B, 61, D)

        for layer in self.encoder:
            x_vis = layer(x_vis)
        x_vis = self.encoder_norm(x_vis)

        x_dec       = self.decoder_embed(x_vis)
        mask_tokens = self.mask_token.expand(B, num_masked, -1)
        x_full      = torch.cat([x_dec[:, 1:], mask_tokens], dim=1)
        x_full      = torch.gather(
            x_full, 1,
            ids_restore.unsqueeze(-1).expand(-1, -1, x_full.shape[-1])
        )
        x_full = torch.cat([x_dec[:, :1], x_full], dim=1) + self.decoder_pos_embed

        for layer in self.decoder:
            x_full = layer(x_full)
        pred = self.decoder_pred(self.decoder_norm(x_full))  # (B, 241, P)

        target = signals.reshape(
            B, self.num_leads,
            self.patch_embed.num_patches_per_lead,
            self.patch_embed.patch_size
        ).reshape(B, -1, self.patch_embed.patch_size)  # (B, 240, P)

        mask = torch.zeros(B, N, device=signals.device)
        mask.scatter_(1, ids_shuffle[:, :num_masked], 1)

        loss = ((pred[:, 1:] - target) ** 2)
        loss = (loss * mask.unsqueeze(-1)).sum() / mask.sum() / self.patch_embed.patch_size

        return loss, pred, mask


# ── Checkpoint Manager ────────────────────────────────────────────
class BatchCheckpointManager:
    """
    Batch-level checkpointing — never loses more than
    checkpoint_every_batches (~10-15 min) of work.

    Two files saved to Drive:
      stmem_1d_checkpoint.pt   full rolling checkpoint
      stmem_1d_pretrained.pt   encoder-only best-loss snapshot
    """

    def __init__(self, checkpoint_dir):
        self.ckpt_path   = Path(checkpoint_dir) / 'stmem_1d_checkpoint.pt'
        self.best_path   = Path(checkpoint_dir) / 'stmem_1d_pretrained.pt'
        self.loss_log    = []
        self.interrupted = False
        Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

        def _handler(sig, frame):
            print('\n⚠️  Ctrl+C detected — saving checkpoint before exit...')
            self.interrupted = True
        signal.signal(signal.SIGINT, _handler)

    def save(self, epoch, batch_idx, total_batches,
             model, optimizer, scheduler, scaler,
             loss, best_loss, is_best=False, epoch_complete=False):

        ckpt = {
            'epoch'               : epoch,
            'batch_idx'           : batch_idx,
            'total_batches'       : total_batches,
            'epoch_complete'      : epoch_complete,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict'   : scaler.state_dict() if scaler else None,
            'loss'                : loss,
            'best_loss'           : best_loss,
            'loss_log'            : self.loss_log,
            'timestamp'           : datetime.now().isoformat(),
        }
        # Atomic write — no corrupt checkpoint on Drive if session drops mid-save
        tmp = self.ckpt_path.with_suffix('.tmp')
        torch.save(ckpt, tmp)
        tmp.replace(self.ckpt_path)

        status = 'epoch complete ✅' if epoch_complete \
                 else f'batch {batch_idx}/{total_batches}'
        print(f'  💾 Saved — epoch {epoch+1} | {status} | loss {loss:.4f}')

        if is_best:
            # Encoder-only — key set matches load_stmem_pretrained() in vit_1d_fm.py
            torch.save({
                'epoch': epoch,
                'loss' : loss,
                'model_state_dict': {
                    'patch_embed' : model.patch_embed.state_dict(),
                    'cls_token'   : model.cls_token.data,
                    'pos_embed'   : model.pos_embed.data,
                    'sep_tokens'  : model.sep_tokens.data,
                    'encoder'     : model.encoder.state_dict(),
                    'encoder_norm': model.encoder_norm.state_dict(),
                }
            }, self.best_path)
            print(f'  🏆 New best: {loss:.4f} → stmem_1d_pretrained.pt')

    def load(self, model, optimizer, scheduler, scaler, device):
        if not self.ckpt_path.exists():
            print('  ℹ️  No checkpoint — starting fresh')
            return 0, 0, float('inf')

        print(f'  📂 Loading: {self.ckpt_path}')
        # CPU-first prevents optimizer device mismatch after notebook restart
        ckpt = torch.load(self.ckpt_path, map_location='cpu', weights_only=False)

        model.load_state_dict(ckpt['model_state_dict'])
        model.to(device)
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        self.loss_log = ckpt.get('loss_log', [])
        epoch    = ckpt['epoch']
        batch    = ckpt.get('batch_idx', 0)
        best     = ckpt['best_loss']
        complete = ckpt.get('epoch_complete', True)
        ts       = ckpt.get('timestamp', '?')

        print(f'  ⏱️  Saved at  : {ts}')
        print(f'  📉 Best loss : {best:.4f}')

        if complete:
            print(f'  ▶️  Epoch {epoch+1} complete → starting epoch {epoch+2}')
            return epoch + 1, 0, best
        else:
            print(f'  ▶️  Epoch {epoch+1} interrupted at batch {batch} → resuming batch {batch+1}')
            return epoch, batch + 1, best


print('✅ All classes defined')
print('   Architecture : original (sep_tokens saved but not used in forward)')
print('   pos_embed    : (1, 241, 768) — CLS + 240 patches')
print('   File listing : subprocess find, no timeout')
print('   Checkpoint   : CPU-first load, atomic write')

In [ ]:
import threading

def colab_keepalive(interval_sec=1800):
    """Heartbeat every 30 min to prevent Colab idle disconnect."""
    def _beat():
        count = 0
        while True:
            time.sleep(interval_sec)
            count += 1
            print(f'  💓 Keepalive #{count} ({datetime.now().strftime("%H:%M:%S")})')
    threading.Thread(target=_beat, daemon=True).start()
    print('✅ Keepalive started (every 30 min)')


colab_keepalive()


def train_stmem(cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f'\n{"="*65}')
    print('🚀  ST-MEM 1D Pretraining — ChagaSight')
    print(f'{"="*65}')
    if device.type == 'cuda':
        print(f'GPU  : {torch.cuda.get_device_name(0)}')
        print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'AMP  : {cfg["use_amp"]}  |  Epochs: {cfg["epochs"]}  |  Batch: {cfg["batch_size"]}')
    print(f'{"="*65}\n')

    # ── Dataset ──────────────────────────────────────────────────
    print('📂 Loading dataset...')
    print('   (code15 path collection: ~2-3 min over Drive FUSE — normal)')
    dataset = STMEMSignalDataset(cfg['data_dir'], subset=cfg['subset'])

    loader = DataLoader(
        dataset,
        batch_size=cfg['batch_size'],
        shuffle=True,
        num_workers=cfg['num_workers'],
        pin_memory=(device.type == 'cuda'),
        drop_last=True,
        persistent_workers=(cfg['num_workers'] > 0),
        prefetch_factor=2 if cfg['num_workers'] > 0 else None,
    )
    total_batches = len(loader)
    print(f'  {total_batches:,} batches per epoch\n')

    # ── Model ────────────────────────────────────────────────────
    model = STMEM1D(mask_ratio=cfg['mask_ratio']).to(device)
    total = sum(p.numel() for p in model.parameters())
    enc   = sum(p.numel() for n, p in model.named_parameters() if 'decoder' not in n)
    print(f'[MODEL] Total: {total:,}  Encoder: {enc:,}  Decoder: {total-enc:,}\n')

    # ── Optimizer (fused with fallback) ──────────────────────────
    try:
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg['lr'],
            betas=(0.9, 0.95), weight_decay=0.05, fused=True)
        print('⚡ Fused AdamW enabled')
    except TypeError:
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg['lr'],
            betas=(0.9, 0.95), weight_decay=0.05)
        print('ℹ️  Standard AdamW')

    # ── Scheduler ────────────────────────────────────────────────
    def lr_lambda(ep):
        if ep < cfg['warmup_epochs']:
            return (ep + 1) / cfg['warmup_epochs']
        p = (ep - cfg['warmup_epochs']) / max(cfg['epochs'] - cfg['warmup_epochs'], 1)
        return 0.5 * (1.0 + np.cos(np.pi * p))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # ── AMP Scaler (with API fallback) ───────────────────────────
    scaler = None
    if cfg['use_amp'] and device.type == 'cuda':
        try:
            scaler = torch.amp.GradScaler('cuda')
            print('✅ AMP: torch.amp.GradScaler')
        except (TypeError, AttributeError):
            scaler = torch.cuda.amp.GradScaler()
            print('✅ AMP: torch.cuda.amp.GradScaler (fallback)')

    # ── Load checkpoint → auto-resume ────────────────────────────
    ckpt_mgr = BatchCheckpointManager(cfg['checkpoint_dir'])
    start_epoch, start_batch, best_loss = ckpt_mgr.load(
        model, optimizer, scheduler, scaler, device)

    if start_epoch >= cfg['epochs']:
        print(f'✅ Already complete ({cfg["epochs"]} epochs, best loss: {best_loss:.4f})')
        return

    print(f'\n▶️  Epoch {start_epoch+1}/{cfg["epochs"]}, batch {start_batch}/{total_batches}\n')
    t0 = time.time()

    # ── Training loop ─────────────────────────────────────────────
    for epoch in range(start_epoch, cfg['epochs']):
        model.train()
        epoch_loss, n_done = 0.0, 0
        t_ep = time.time()

        pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{cfg["epochs"]}')

        for batch_idx, signals in enumerate(pbar):

            # Skip already-done batches on resume
            if epoch == start_epoch and batch_idx < start_batch:
                if batch_idx % 500 == 0:
                    pbar.set_postfix({'⏩': f'→{start_batch}'})
                continue

            signals = signals.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            try:
                if scaler:
                    with torch.amp.autocast('cuda'):
                        loss, _, _ = model(signals)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss, _, _ = model(signals)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
            except RuntimeError as e:
                if 'autocast' in str(e):
                    with torch.cuda.amp.autocast():
                        loss, _, _ = model(signals)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    raise

            epoch_loss += loss.item()
            n_done     += 1
            avg         = epoch_loss / n_done

            pbar.set_postfix({
                'loss' : f'{loss.item():.4f}',
                'avg'  : f'{avg:.4f}',
                'best' : f'{best_loss:.4f}',
                'lr'   : f'{optimizer.param_groups[0]["lr"]:.2e}',
            })

            # Batch-level checkpoint
            if (batch_idx + 1) % cfg['checkpoint_every_batches'] == 0:
                is_best = avg < best_loss
                if is_best: best_loss = avg
                ckpt_mgr.loss_log.append({'epoch': epoch, 'batch': batch_idx, 'loss': avg})
                ckpt_mgr.save(epoch, batch_idx, total_batches,
                              model, optimizer, scheduler, scaler,
                              avg, best_loss, is_best=is_best, epoch_complete=False)

            # Graceful Ctrl+C
            if ckpt_mgr.interrupted:
                avg = epoch_loss / max(n_done, 1)
                ckpt_mgr.loss_log.append({'epoch': epoch, 'batch': batch_idx, 'loss': avg})
                ckpt_mgr.save(epoch, batch_idx, total_batches,
                              model, optimizer, scheduler, scaler,
                              avg, best_loss, is_best=False, epoch_complete=False)
                print('\n🛑 Stopped cleanly. Re-run Cell 6 to resume.')
                return

        # ── End of epoch ─────────────────────────────────────────
        start_batch = 0
        scheduler.step()

        avg     = epoch_loss / max(n_done, 1)
        is_best = avg < best_loss
        if is_best: best_loss = avg

        elapsed = time.time() - t0
        eta_h   = (elapsed / (epoch - start_epoch + 1)) * (cfg['epochs'] - epoch - 1) / 3600

        ckpt_mgr.loss_log.append({'epoch': epoch, 'batch': 'end', 'loss': avg})
        ckpt_mgr.save(epoch, total_batches - 1, total_batches,
                      model, optimizer, scheduler, scaler,
                      avg, best_loss, is_best=is_best, epoch_complete=True)

        print(f'[EPOCH {epoch+1:02d}/{cfg["epochs"]}]  '
              f'loss={avg:.4f}  '
              f'lr={optimizer.param_groups[0]["lr"]:.2e}  '
              f'time={(time.time()-t_ep)/60:.1f}min  '
              f'ETA={eta_h:.1f}h')

        # Milestone snapshot
        if (epoch + 1) % cfg['save_every'] == 0:
            ms = Path(cfg['checkpoint_dir']) / f'stmem_1d_epoch{epoch+1}.pt'
            torch.save({
                'epoch': epoch, 'loss': avg,
                'model_state_dict': {
                    'patch_embed' : model.patch_embed.state_dict(),
                    'cls_token'   : model.cls_token.data,
                    'pos_embed'   : model.pos_embed.data,
                    'sep_tokens'  : model.sep_tokens.data,
                    'encoder'     : model.encoder.state_dict(),
                    'encoder_norm': model.encoder_norm.state_dict(),
                }
            }, ms)
            print(f'  📌 Milestone: stmem_1d_epoch{epoch+1}.pt')

    total_h = (time.time() - t0) / 3600
    print(f'\n🎉 Done!  {total_h:.1f}h  |  Best loss: {best_loss:.4f}')
    print(f'   → {Path(cfg["checkpoint_dir"]) / "stmem_1d_pretrained.pt"}')


print('✅ Training function ready')

In [ ]:
# ──────────────────────────────────────────────────────────────
#  AFTER A COLAB DISCONNECT:
#  Re-run Cells 1 → 2 → 4 → 5 → this cell
#  Training auto-resumes from last saved batch
#
#  Note: dataset init takes ~2-3 min for code15 path collection
#  This is normal — it only happens once per session
# ──────────────────────────────────────────────────────────────
train_stmem(CONFIG)

In [ ]:
from pathlib import Path
import torch

CKPT_DIR = Path(CONFIG['checkpoint_dir'])

print('=' * 65)
print('📊  CHECKPOINT STATUS')
print('=' * 65)

for fname, label in [
    ('stmem_1d_checkpoint.pt', 'Rolling checkpoint'),
    ('stmem_1d_pretrained.pt', 'Best encoder (for fine-tuning)'),
]:
    f = CKPT_DIR / fname
    if not f.exists():
        print(f'\n❌ {label}: not found')
        continue

    c = torch.load(f, map_location='cpu', weights_only=False)
    print(f'\n✅ {label}')
    print(f'   Epoch : {c["epoch"]+1}  |  Loss : {c["loss"]:.4f}')
    if 'best_loss' in c:
        print(f'   Best  : {c["best_loss"]:.4f}')
    if 'model_state_dict' in c and 'encoder' in c.get('model_state_dict', {}):
        print(f'   Keys  : {list(c["model_state_dict"].keys())}')
        pos = c['model_state_dict']['pos_embed']
        compat = '✅ 241=CLS+240' if pos.shape[1] == 241 else '⚠️  unexpected'
        print(f'   pos_embed : {tuple(pos.shape)}  {compat}')
    print(f'   Size  : {f.stat().st_size/1e6:.1f} MB')
    status = c.get('epoch_complete', True)
    print(f'   Status: {"complete ✅" if status else f"interrupted at batch {c.get(chr(98)+chr(97)+chr(116)+chr(99)+chr(104)+chr(95)+chr(105)+chr(100))}"}')

milestones = sorted(CKPT_DIR.glob('stmem_1d_epoch*.pt'))
if milestones:
    print(f'\n📌 Milestones ({len(milestones)}):')
    for m in milestones:
        c = torch.load(m, map_location='cpu', weights_only=False)
        print(f'   {m.name:<35} loss={c["loss"]:.4f}')

print('\n' + '=' * 65)
print('📥  Next steps:')
print('   1. Download stmem_1d_pretrained.pt from Drive')
print('   2. Copy → local checkpoints/stmem_1d_pretrained.pt')
print('   3. Used automatically by:')
print('      model.load_stmem_pretrained("checkpoints/stmem_1d_pretrained.pt")')
print('=' * 65)